# Primera versión del detector de aumentos inusuales

## Objetivo

Identificar aumentos mensuales del paro registrado que superen una
referencia histórica específica de cada municipio y mes del año.

## Método inicial

Para cada observación utilizaremos los cambios absolutos del mismo mes
en los cinco años anteriores.

- Referencia: mediana de los cinco cambios históricos.
- Dispersión: desviación absoluta mediana (MAD).
- Señal experimental: aumento efectivo del paro que supere la referencia
  más tres veces la MAD escalada.

El cambio actual queda excluido de su propia referencia.

## Limitaciones iniciales

Cinco observaciones son pocas para estimar la variabilidad con precisión.
Esta regla es un punto de partida, no un detector validado.

El método considera el mes del año, pero no modeliza explícitamente
la tendencia reciente ni los cambios en el volumen de paro.

Los casos sin historial suficiente o con MAD igual a cero quedarán
marcados como no evaluables con esta regla, no como ausencia de alerta.

Los porcentajes se mostrarán como información complementaria.

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Localizar la raíz tanto si el notebook se ejecuta desde
# la carpeta del proyecto como desde notebooks.
RAIZ = Path.cwd()

if not (RAIZ / "datos").is_dir():
    RAIZ = RAIZ.parent

RUTA_PARO = (
    RAIZ / "datos" / "procesados" / "paro_total_mensual.csv"
)

if not RUTA_PARO.is_file():
    raise FileNotFoundError(
        f"No se encuentra el archivo: {RUTA_PARO}"
    )

paro = pd.read_csv(
    RUTA_PARO,
    sep=";",
    encoding="utf-8-sig",
    parse_dates=["fecha"],
)

paro = (
    paro.sort_values(["municipio", "fecha"])
    .reset_index(drop=True)
)

print("Archivo:", RUTA_PARO)
print("Filas:", len(paro))
print("Municipios:", paro["municipio"].nunique())
print("Primera fecha:", paro["fecha"].min())
print("Última fecha:", paro["fecha"].max())

display(paro.head())

Archivo: c:\Users\romendozaga_bec\Proyectos\pulso-laboral-madrid\datos\procesados\paro_total_mensual.csv
Filas: 46898
Municipios: 179
Primera fecha: 2005-01-01 00:00:00
Última fecha: 2026-10-01 00:00:00


,municipio,fecha,paro_total
0,"Acebeda, La",2005-01-01,3
1,"Acebeda, La",2005-02-01,2
2,"Acebeda, La",2005-03-01,2
3,"Acebeda, La",2005-04-01,2
4,"Acebeda, La",2005-05-01,2


In [9]:
columnas_necesarias = {"municipio", "fecha", "paro_total"}

if not columnas_necesarias.issubset(paro.columns):
    raise ValueError("Faltan columnas necesarias.")

if paro[list(columnas_necesarias)].isna().any().any():
    raise ValueError("Hay valores ausentes en columnas esenciales.")

if paro.duplicated(["municipio", "fecha"]).any():
    raise ValueError("Hay municipios y fechas duplicados.")

paro["paro_total"] = pd.to_numeric(
    paro["paro_total"], errors="raise"
)

if (paro["paro_total"] < 0).any():
    raise ValueError("Hay valores negativos de paro.")

if (paro["paro_total"] % 1 != 0).any():
    raise ValueError("Hay recuentos de personas no enteros.")

if not paro["fecha"].dt.is_month_start.all():
    raise ValueError("Las fechas deben representar el inicio del mes.")

for municipio, grupo in paro.groupby("municipio"):
    fechas_esperadas = pd.date_range(
        grupo["fecha"].min(),
        grupo["fecha"].max(),
        freq="MS",
    )

    if not pd.DatetimeIndex(grupo["fecha"]).equals(
        fechas_esperadas
    ):
        raise ValueError(
            f"Faltan meses o hay fechas incorrectas en {municipio}."
        )

print("Comprobaciones superadas.")

Comprobaciones superadas.


In [10]:
paro["paro_mes_anterior"] = (
    paro.groupby("municipio")["paro_total"].shift(1)
)

paro["cambio_mensual"] = (
    paro["paro_total"] - paro["paro_mes_anterior"]
)

base_porcentaje = paro["paro_mes_anterior"].where(
    paro["paro_mes_anterior"] > 0
)

paro["variacion_mensual_pct"] = (
    100 * paro["cambio_mensual"] / base_porcentaje
)

paro["mes"] = paro["fecha"].dt.month

display(
    paro[
        [
            "municipio",
            "fecha",
            "paro_total",
            "paro_mes_anterior",
            "cambio_mensual",
            "variacion_mensual_pct",
        ]
    ].head(15)
)

,municipio,fecha,paro_total,paro_mes_anterior,cambio_mensual,variacion_mensual_pct
0,"Acebeda, La",2005-01-01,3,NaN,NaN,NaN
1,"Acebeda, La",2005-02-01,2,3.0,-1.0,-33.333333
2,"Acebeda, La",2005-03-01,2,2.0,0.0,0.000000
3,"Acebeda, La",2005-04-01,2,2.0,0.0,0.000000
4,"Acebeda, La",2005-05-01,2,2.0,0.0,0.000000
5,"Acebeda, La",2005-06-01,2,2.0,0.0,0.000000
6,"Acebeda, La",2005-07-01,0,2.0,-2.0,-100.000000
7,"Acebeda, La",2005-08-01,1,0.0,1.0,NaN
8,"Acebeda, La",2005-09-01,0,1.0,-1.0,-100.000000
9,"Acebeda, La",2005-10-01,0,0.0,0.0,NaN


In [11]:
VENTANA_ANOS = 5
MIN_OBSERVACIONES = 5
UMBRAL = 3.0


def calcular_mad(valores):
    valores = np.asarray(valores, dtype=float)
    mediana = np.median(valores)
    return np.median(np.abs(valores - mediana))


# Dentro de cada municipio y mes, las observaciones están
# ordenadas por año. shift(1) excluye el dato actual.
grupos = paro.groupby(["municipio", "mes"])["cambio_mensual"]

paro["n_historico"] = grupos.transform(
    lambda serie: (
        serie.shift(1)
        .rolling(VENTANA_ANOS, min_periods=1)
        .count()
    )
).fillna(0).astype(int)

paro["referencia_cambio"] = grupos.transform(
    lambda serie: (
        serie.shift(1)
        .rolling(
            VENTANA_ANOS,
            min_periods=MIN_OBSERVACIONES,
        )
        .median()
    )
)

paro["mad_cambio"] = grupos.transform(
    lambda serie: (
        serie.shift(1)
        .rolling(
            VENTANA_ANOS,
            min_periods=MIN_OBSERVACIONES,
        )
        .apply(calcular_mad, raw=True)
    )
)

paro["escala_robusta"] = 1.4826 * paro["mad_cambio"]

print("Referencias históricas calculadas.")

Referencias históricas calculadas.


In [12]:
historial_suficiente = (
    paro["n_historico"] >= MIN_OBSERVACIONES
)

evaluable = (
    historial_suficiente
    & paro["referencia_cambio"].notna()
    & paro["escala_robusta"].gt(0)
    & paro["cambio_mensual"].notna()
)

paro["desviacion_personas"] = (
    paro["cambio_mensual"] - paro["referencia_cambio"]
)

paro["puntuacion_robusta"] = (
    paro["desviacion_personas"]
    / paro["escala_robusta"].where(evaluable)
)

paro["limite_superior"] = (
    paro["referencia_cambio"]
    + UMBRAL * paro["escala_robusta"]
).where(evaluable)

# Booleano con valores ausentes:
# no evaluable no equivale a False.
paro["alerta_experimental"] = pd.Series(
    pd.NA,
    index=paro.index,
    dtype="boolean",
)

paro.loc[evaluable, "alerta_experimental"] = (
    paro.loc[evaluable, "cambio_mensual"].gt(0)
    & paro.loc[evaluable, "puntuacion_robusta"].gt(UMBRAL)
)

paro["estado_detector"] = "historial_insuficiente"

paro.loc[
    historial_suficiente & paro["escala_robusta"].eq(0),
    "estado_detector",
] = "dispersion_cero"

paro.loc[evaluable, "estado_detector"] = "sin_alerta"

paro.loc[
    paro["alerta_experimental"].fillna(False),
    "estado_detector",
] = "alerta_experimental"

display(paro["estado_detector"].value_counts())

estado_detector
sin_alerta                31880
historial_insuficiente    10919
dispersion_cero            2240
alerta_experimental        1859
Name: count, dtype: int64

In [13]:
MUNICIPIO = "Madrid"
FECHA = pd.Timestamp("2026-10-01")

historico_ejemplo = paro.loc[
    paro["municipio"].eq(MUNICIPIO)
    & paro["mes"].eq(FECHA.month)
    & paro["fecha"].lt(FECHA)
    & paro["fecha"].ge(
        FECHA - pd.DateOffset(years=VENTANA_ANOS)
    ),
    ["fecha", "paro_total", "cambio_mensual"],
]

columnas_resultado = [
    "municipio",
    "fecha",
    "paro_total",
    "cambio_mensual",
    "variacion_mensual_pct",
    "n_historico",
    "referencia_cambio",
    "mad_cambio",
    "limite_superior",
    "puntuacion_robusta",
    "estado_detector",
]

resultado_ejemplo = paro.loc[
    paro["municipio"].eq(MUNICIPIO)
    & paro["fecha"].eq(FECHA),
    columnas_resultado,
]

print("Datos históricos utilizados:")
display(historico_ejemplo)

print("Resultado del mes evaluado:")
display(resultado_ejemplo)

Datos históricos utilizados:


,fecha,paro_total,cambio_mensual
20375,2021-10-01,190511,-5282.0
20387,2022-10-01,146174,-91.0
20399,2023-10-01,143534,61.0
20411,2024-10-01,138984,233.0
20423,2025-10-01,134318,1495.0


Resultado del mes evaluado:


,municipio,fecha,paro_total,cambio_mensual,variacion_mensual_pct,n_historico,referencia_cambio,mad_cambio,limite_superior,puntuacion_robusta,estado_detector
20435,Madrid,2026-10-01,138652,2130.0,1.560188,5,61.0,172.0,826.0216,8.113496,alerta_experimental


In [15]:
ultima_fecha = paro["fecha"].max()

ultimo_mes = paro.loc[
    paro["fecha"].eq(ultima_fecha)
].copy()

print("Mes evaluado:", ultima_fecha.strftime("%Y-%m"))

print("\nEstado de los municipios:")
display(ultimo_mes["estado_detector"].value_counts())

alertas = ultimo_mes.loc[
    ultimo_mes["alerta_experimental"].fillna(False),
    columnas_resultado,
].sort_values(
    "puntuacion_robusta",
    ascending=False,
)

print("\nAlertas experimentales:")
display(alertas.head(20))

print("\nCasos no evaluables:")
display(
    ultimo_mes.loc[
        ultimo_mes["estado_detector"].isin(
            ["historial_insuficiente", "dispersion_cero"]
        ),
        [
            "municipio",
            "paro_total",
            "cambio_mensual",
            "n_historico",
            "mad_cambio",
            "estado_detector",
        ],
    ].head(20)
)

Mes evaluado: 2026-10

Estado de los municipios:


estado_detector
sin_alerta             157
alerta_experimental     15
dispersion_cero          7
Name: count, dtype: int64


Alertas experimentales:


,municipio,fecha,paro_total,cambio_mensual,variacion_mensual_pct,n_historico,referencia_cambio,mad_cambio,limite_superior,puntuacion_robusta,estado_detector
17553,Guadarrama,2026-10-01,693,44.0,6.779661,5,16.0,1.0,20.4478,18.885741,alerta_experimental
33797,San Martín de la Vega,2026-10-01,1158,40.0,3.577818,5,-13.0,2.0,-4.1044,17.874005,alerta_experimental
20435,Madrid,2026-10-01,138652,2130.0,1.560188,5,61.0,172.0,826.0216,8.113496,alerta_experimental
24365,Navalagamella,2026-10-01,191,19.0,11.046512,5,-5.0,2.0,3.8956,8.093889,alerta_experimental
36155,Soto del Real,2026-10-01,355,31.0,9.567901,5,3.0,3.0,16.3434,6.295247,alerta_experimental
27509,Perales de Tajuña,2026-10-01,209,15.0,7.731959,5,-3.0,2.0,5.8956,6.070417,alerta_experimental
15457,Fuentidueña de Tajo,2026-10-01,157,14.0,9.790210,5,5.0,1.0,9.4478,6.070417,alerta_experimental
15719,Galapagar,2026-10-01,1548,59.0,3.962391,5,1.0,7.0,32.1346,5.588638,alerta_experimental
785,Alameda del Valle,2026-10-01,20,9.0,81.818182,5,1.0,1.0,5.4478,5.395926,alerta_experimental
42705,"Vellón, El",2026-10-01,140,10.0,7.692308,5,2.0,1.0,6.4478,5.395926,alerta_experimental



Casos no evaluables:


,municipio,paro_total,cambio_mensual,n_historico,mad_cambio,estado_detector
3929,"Atazar, El",4,-4.0,5,0.0,dispersion_cero
6025,Braojos,5,-1.0,5,0.0,dispersion_cero
25937,Orusco de Tajuña,95,3.0,5,0.0,dispersion_cero
30129,Quijorna,154,-10.0,5,0.0,dispersion_cero
31963,Robregordo,5,-1.0,5,0.0,dispersion_cero
35893,Somosierra,3,1.0,5,0.0,dispersion_cero
39299,Valdaracete,39,2.0,5,0.0,dispersion_cero


### Interpretación del detector inicial: octubre de 2026

#### Qué estamos midiendo

El detector compara el cambio mensual del paro registrado de cada
municipio con los cambios del mismo mes en los cinco años anteriores.

La referencia es la mediana histórica. La dispersión se mide mediante
la desviación absoluta mediana (MAD), multiplicada por 1,4826.

Se señala una alerta experimental cuando se cumplen simultáneamente
dos condiciones:

1. El paro registrado aumenta respecto al mes anterior.
2. El cambio supera la referencia más tres veces la MAD escalada.

El mes evaluado no interviene en el cálculo de su propia referencia.
La puntuación obtenida no es una probabilidad ni una medida de gravedad
social. El umbral de tres es provisional y está pendiente de evaluación.

#### Resultado para Madrid

En octubre de 2026, Madrid registra un incremento mensual de
2.130 personas, equivalente al 1,56 %.

Los cambios de octubre de 2021 a 2025 son −5.282, −91, +61, +233
y +1.495 personas. Su mediana es +61 y su MAD es 172.

Con estos valores, el límite superior del cambio mensual es
aproximadamente +826,022 personas. El aumento observado supera
ese límite y obtiene una puntuación de 8,113, por lo que se clasifica
como alerta experimental.

Sin embargo, los cinco cambios históricos muestran una progresión
ascendente. La referencia basada en su mediana no incorpora ese orden
temporal. Será necesario comprobar si considerar la evolución reciente
modifica la interpretación de esta alerta.

#### Resultado del conjunto de municipios

De los 179 municipios del último mes:

- 157 no activan la alerta.
- 15 activan una alerta experimental.
- 7 quedan sin evaluar con esta regla porque su MAD es cero.

Se evalúan, por tanto, 172 municipios. Las 15 alertas representan
aproximadamente el 8,72 % de los municipios evaluables. Esta cifra
describe la frecuencia de alertas del mes, no la precisión del detector.

La ausencia de alerta no demuestra que la evolución laboral sea
favorable; únicamente indica que no se cumplen las condiciones
de esta regla.

#### Limitaciones observadas

- **Historial reducido:** cinco cambios aportan poca información para
  estimar de forma estable la referencia y la dispersión.

- **MAD pequeña:** puede generar puntuaciones muy altas. Guadarrama
  presenta una puntuación de 18,886 con un aumento de 44 personas,
  una referencia de 16 y una MAD de 1. Esta puntuación no implica
  una mayor gravedad que la de municipios con aumentos más numerosos.

- **MAD cero:** impide calcular la puntuación, pero no implica que todos
  los cambios históricos sean idénticos. Estos municipios se conservan
  como no evaluables y requieren un tratamiento específico.

- **Evolución reciente:** el método compara el mismo mes del año,
  pero no ajusta explícitamente la tendencia ni los cambios en
  el volumen de paro.

- **Interpretación:** las alertas son señales para revisión. No
  identifican causas ni permiten concluir que exista un error de datos.

#### Próximas comprobaciones

Se evaluará el comportamiento del detector en meses históricos,
la sensibilidad a la ventana temporal y al umbral, y la influencia
de las dispersiones pequeñas o nulas.

También se comprobará si incorporar la evolución reciente mejora
la referencia. Las comparaciones utilizarán únicamente información
anterior al mes evaluado.

Hasta completar estas comprobaciones, los resultados se presentarán
como alertas experimentales.